<a href="https://colab.research.google.com/github/Navjotkhatri/Specialized-LLM-Bot-Using-Pre-Trained-Models/blob/main/Specialized_LLM_Bot_Using_Pre_Trained_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Specialized LLM Bot Using Pre-Trained Models**

## Project Goal

The primary goal of this capstone project is to develop an industry-specific Large Language Model (LLM) Bot using pre-trained models from platforms such as Hugging Face. Students will be tasked with selecting one industry from a provided list, gathering relevant data, fine-tuning a pre-trained LLM, and demonstrating the bot's capability to engage users effectively by providing accurate and contextually appropriate responses.

**Industry - Technology and IT**

## Data Collection - Web Scraping

In [54]:
!pip install requests beautifulsoup4 pandas lxml

website crawler

In [55]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import pandas as pd
import time

BASE_URL = "https://www.kreativekudi.com/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/151.0.0.0 Safari/537.36"
}

visited = set()
pages = []

def is_internal(url):
    base_domain = urlparse(BASE_URL).netloc
    return urlparse(url).netloc == base_domain


def scrape_page(url):

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=15
        )

        if response.status_code != 200:
            print(f"Failed: {url} | Status: {response.status_code}")
            return

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove unnecessary elements
        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        title = soup.title.get_text(strip=True) if soup.title else ""

        headings = [
            h.get_text(" ", strip=True)
            for h in soup.find_all(["h1", "h2", "h3", "h4"])
        ]

        paragraphs = [
            p.get_text(" ", strip=True)
            for p in soup.find_all("p")
            if p.get_text(strip=True)
        ]

        content = "\n".join(paragraphs)

        pages.append({
            "url": url,
            "title": title,
            "headings": " | ".join(headings),
            "content": content
        })

        print(f"Scraped: {url}")

        # Find internal links
        for link in soup.find_all("a", href=True):

            next_url = urljoin(url, link["href"])

            # Remove fragments
            next_url = next_url.split("#")[0]

            if (
                is_internal(next_url)
                and next_url not in visited
                and next_url.startswith("https://")
            ):
                visited.add(next_url)
                scrape_page(next_url)

        time.sleep(0.5)

    except Exception as e:
        print(f"Error: {url}")
        print(e)


visited.add(BASE_URL)
scrape_page(BASE_URL)

print("\nTotal pages scraped:", len(pages))

Scraped: https://www.kreativekudi.com/
Scraped: https://www.kreativekudi.com/index.html
Scraped: https://www.kreativekudi.com/Institute.html
Scraped: https://www.kreativekudi.com/about.html
Scraped: https://www.kreativekudi.com/portfolio.html
Scraped: https://www.kreativekudi.com/articles.html
Scraped: https://www.kreativekudi.com/Owner.html
Scraped: https://www.kreativekudi.com/contact.html
Scraped: https://www.kreativekudi.com/article1.html
Scraped: https://www.kreativekudi.com/article2.html
Scraped: https://www.kreativekudi.com/article3.html
Scraped: https://www.kreativekudi.com/personal logo/PRIVACY POLICY of kreative kudi[1].pdf

Total pages scraped: 12


In [56]:
df = pd.DataFrame(pages)

df.head()

,url,title,headings,content
0,https://www.kreativekudi.com/,Kreative kudi-Transforming Ideas into Visual M...,Transform Your Online Presence | Professional ...,"In the world of Internet Customer Service, it’..."
1,https://www.kreativekudi.com/index.html,Kreative kudi-Transforming Ideas into Visual M...,Transform Your Online Presence | Professional ...,"In the world of Internet Customer Service, it’..."
2,https://www.kreativekudi.com/Institute.html,Your Graphic Designing Institute | Hands-On Le...,Become a certified graphic designer | The Plet...,"Learn skills to become job ready, Best Coachin..."
3,https://www.kreativekudi.com/about.html,Elevate Your Brand with Graphic Design and Dig...,Our dedicated team of creatives is bursting\r\...,"At the intersection of art, technology and bus..."
4,https://www.kreativekudi.com/portfolio.html,Explore our company portfolio: A showcase of o...,Amazing Works,Creativity involves breaking out of expected &...


In [57]:
df.to_csv(
    "kreative_kudi_website_data.csv",
    index=False,
    encoding="utf-8"
)

print("Saved successfully!")
print("Rows:", len(df))

Saved successfully!
Rows: 12


In [58]:
print("Number of pages:", len(df))

for i, row in df.iterrows():
    print("\n" + "="*80)
    print("PAGE:", row["url"])
    print("TITLE:", row["title"])
    print("HEADINGS:", row["headings"][:500])
    print("CONTENT:", row["content"][:1000])

Number of pages: 12

PAGE: https://www.kreativekudi.com/
TITLE: Kreative kudi-Transforming Ideas into Visual Masterpieces
HEADINGS: Transform Your Online Presence | Professional Logo Design Solutions | Maximizing Business Growth | We are a One Stop Digital Solution Company | What we did and achieved in our Digital journey | 1200 | 840 | 25 | What Client's Say? | Our Clients | World News. | about your next project
CONTENT: In the world of Internet Customer Service, it’s important to remember your
                                            competitor is only one mouse click away.
Logos are the graphic extension of the internal realities of a company..
The Internet makes money for you when you
                                            build something that is real and when it matters to people!
Click on this tab to open WhatsApp and start a chat.
Welcome to Kreative Kudi Digital Marketing Company and Institute, where we specialize in
                                transforming your onl

In [59]:
import pandas as pd
import re
from urllib.parse import urlparse

# Load scraped data
df = pd.read_csv("kreative_kudi_website_data.csv")

print("Original shape:", df.shape)
display(df.head())

Original shape: (12, 4)


,url,title,headings,content
0,https://www.kreativekudi.com/,Kreative kudi-Transforming Ideas into Visual M...,Transform Your Online Presence | Professional ...,"In the world of Internet Customer Service, it’..."
1,https://www.kreativekudi.com/index.html,Kreative kudi-Transforming Ideas into Visual M...,Transform Your Online Presence | Professional ...,"In the world of Internet Customer Service, it’..."
2,https://www.kreativekudi.com/Institute.html,Your Graphic Designing Institute | Hands-On Le...,Become a certified graphic designer | The Plet...,"Learn skills to become job ready, Best Coachin..."
3,https://www.kreativekudi.com/about.html,Elevate Your Brand with Graphic Design and Dig...,Our dedicated team of creatives is bursting\r\...,"At the intersection of art, technology and bus..."
4,https://www.kreativekudi.com/portfolio.html,Explore our company portfolio: A showcase of o...,Amazing Works,Creativity involves breaking out of expected &...


In [60]:
# Remove rows containing PDF or non-HTML resources
pdf_mask = (
    df["url"].astype(str).str.lower().str.contains(r"\.pdf|/personal", regex=True)
    |
    df["content"].astype(str).str.contains(
        r"%PDF|JFIF|PK\x03\x04",
        regex=True,
        na=False
    )
)

print("Invalid/PDF pages found:", pdf_mask.sum())

df = df[~pdf_mask].copy()

print("After removing invalid pages:", df.shape)

Invalid/PDF pages found: 1
After removing invalid pages: (11, 4)


In [61]:
# Normalize URLs before duplicate removal
def normalize_url(url):
    url = str(url).strip()

    # Remove trailing slash
    url = url.rstrip("/")

    # Treat index.html as homepage
    if url.endswith("/index.html"):
        url = url[:-10]

    return url


df["normalized_url"] = df["url"].apply(normalize_url)

# Remove duplicate pages
df = df.drop_duplicates(
    subset=["normalized_url"],
    keep="first"
).copy()

print("After duplicate removal:", df.shape)

After duplicate removal: (11, 5)


In [62]:
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove escaped characters
    text = text.replace("\\n", " ")
    text = text.replace("\\t", " ")
    text = text.replace("\\r", " ")

    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove repeated spaces
    text = re.sub(r" +", " ", text)

    return text.strip()


df["title"] = df["title"].apply(clean_text)
df["headings"] = df["headings"].apply(clean_text)
df["content"] = df["content"].apply(clean_text)

In [63]:
df["document"] = (
    "Page Title: " + df["title"] +
    "\n\nHeadings: " + df["headings"] +
    "\n\nContent: " + df["content"]
)

In [64]:
df["content_length"] = df["content"].str.len()

df = df[df["content_length"] >= 100].copy()

print("Final shape:", df.shape)

Final shape: (11, 7)


In [65]:
for i, row in df.iterrows():

    print("\n" + "=" * 100)
    print("URL:", row["url"])
    print("TITLE:", row["title"])
    print("CONTENT LENGTH:", row["content_length"])
    print("\nCONTENT:")
    print(row["content"][:1000])


URL: https://www.kreativekudi.com/
TITLE: Kreative kudi-Transforming Ideas into Visual Masterpieces
CONTENT LENGTH: 4381

CONTENT:
In the world of Internet Customer Service, it’s important to remember your competitor is only one mouse click away. Logos are the graphic extension of the internal realities of a company.. The Internet makes money for you when you build something that is real and when it matters to people! Click on this tab to open WhatsApp and start a chat. Welcome to Kreative Kudi Digital Marketing Company and Institute, where we specialize in transforming your online presence and driving exceptional results. With a core focus on SEO, social media management, and content marketing, we are dedicated to enhancing your brand's visibility and attracting targeted traffic to your website. At Kreative Kudi, we are not just experts; we are partners in your online success journey. We achieve this through strategic PPC campaigns and data-driven SEO strategies that deliver measurab

In [66]:
clean_file = "kreative_kudi_cleaned.csv"

df.to_csv(
    clean_file,
    index=False,
    encoding="utf-8"
)

print(f"\nClean dataset saved as: {clean_file}")


Clean dataset saved as: kreative_kudi_cleaned.csv


In [67]:
# Content statistics

df[["title", "content_length"]].sort_values(
    "content_length",
    ascending=False
)

,title,content_length
10,Explore the post-pandemic resurgence of digita...,7748
9,Digital marketing growth in India in 2023-Expl...,6006
0,Kreative kudi-Transforming Ideas into Visual M...,4381
1,Kreative kudi-Transforming Ideas into Visual M...,4381
3,Elevate Your Brand with Graphic Design and Dig...,3966
8,Explore our digital marketing blog,3579
6,Empowering Your Digital Presence | Founder & CEo,2534
2,Your Graphic Designing Institute | Hands-On Le...,2420
5,Stay updated with our insightful digital marke...,1148
4,Explore our company portfolio: A showcase of o...,521


In [68]:
import pandas as pd
import re

# Work on a copy
rag_df = df.copy()

# --------------------------------------------------
# 1. Clean text function
# --------------------------------------------------

def clean_rag_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove escaped characters
    text = text.replace("\\n", " ")
    text = text.replace("\\t", " ")
    text = text.replace("\\r", " ")

    # Remove email/HTML artifacts
    text = re.sub(r"mailto:\S+", "", text)

    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


rag_df["title"] = rag_df["title"].apply(clean_rag_text)
rag_df["headings"] = rag_df["headings"].apply(clean_rag_text)
rag_df["content"] = rag_df["content"].apply(clean_rag_text)


# --------------------------------------------------
# 2. Remove common website boilerplate
# --------------------------------------------------

boilerplate_patterns = [
    r"©\s*Kreative\s*Kudi.*?$",
    r"All rights reserved.*?$",
]

def remove_boilerplate(text):

    for pattern in boilerplate_patterns:
        text = re.sub(
            pattern,
            "",
            text,
            flags=re.IGNORECASE
        )

    return text.strip()


rag_df["content"] = rag_df["content"].apply(remove_boilerplate)


# --------------------------------------------------
# 3. Create the final document
# --------------------------------------------------

rag_df["document"] = (
    "Title: " + rag_df["title"] +
    "\n\n" +
    "Headings: " + rag_df["headings"] +
    "\n\n" +
    "Content: " + rag_df["content"]
)


# --------------------------------------------------
# 4. Remove empty documents
# --------------------------------------------------

rag_df = rag_df[
    rag_df["content"].str.len() > 100
].copy()


# --------------------------------------------------
# 5. Add document ID
# --------------------------------------------------

rag_df = rag_df.reset_index(drop=True)

rag_df["document_id"] = [
    f"KK_DOC_{i+1:03d}"
    for i in range(len(rag_df))
]


# --------------------------------------------------
# 6. Final columns
# --------------------------------------------------

rag_df = rag_df[
    [
        "document_id",
        "url",
        "title",
        "headings",
        "content",
        "document"
    ]
]


print("RAG documents:", len(rag_df))
print("Total characters:", rag_df["document"].str.len().sum())

display(
    rag_df[
        ["document_id", "title", "url"]
    ]
)

RAG documents: 11
Total characters: 39610


,document_id,title,url
0,KK_DOC_001,Kreative kudi-Transforming Ideas into Visual M...,https://www.kreativekudi.com/
1,KK_DOC_002,Kreative kudi-Transforming Ideas into Visual M...,https://www.kreativekudi.com/index.html
2,KK_DOC_003,Your Graphic Designing Institute | Hands-On Le...,https://www.kreativekudi.com/Institute.html
3,KK_DOC_004,Elevate Your Brand with Graphic Design and Dig...,https://www.kreativekudi.com/about.html
4,KK_DOC_005,Explore our company portfolio: A showcase of o...,https://www.kreativekudi.com/portfolio.html
5,KK_DOC_006,Stay updated with our insightful digital marke...,https://www.kreativekudi.com/articles.html
6,KK_DOC_007,Empowering Your Digital Presence | Founder & CEo,https://www.kreativekudi.com/Owner.html
7,KK_DOC_008,Get in touch with us today | Contact us,https://www.kreativekudi.com/contact.html
8,KK_DOC_009,Explore our digital marketing blog,https://www.kreativekudi.com/article1.html
9,KK_DOC_010,Digital marketing growth in India in 2023-Expl...,https://www.kreativekudi.com/article2.html


In [69]:
rag_df.to_csv(
    "kreative_kudi_rag_documents.csv",
    index=False,
    encoding="utf-8"
)

print("RAG dataset saved successfully.")

RAG dataset saved successfully.


In [70]:
print(rag_df.loc[0, "document"])

Title: Kreative kudi-Transforming Ideas into Visual Masterpieces

Headings: Transform Your Online Presence | Professional Logo Design Solutions | Maximizing Business Growth | We are a One Stop Digital Solution Company | What we did and achieved in our Digital journey | 1200 | 840 | 25 | What Client's Say? | Our Clients | World News. | about your next project

Content: In the world of Internet Customer Service, it’s important to remember your competitor is only one mouse click away. Logos are the graphic extension of the internal realities of a company.. The Internet makes money for you when you build something that is real and when it matters to people! Click on this tab to open WhatsApp and start a chat. Welcome to Kreative Kudi Digital Marketing Company and Institute, where we specialize in transforming your online presence and driving exceptional results. With a core focus on SEO, social media management, and content marketing, we are dedicated to enhancing your brand's visibility a

In [71]:
rag_df["document_length"] = rag_df["document"].str.len()

display(
    rag_df[
        ["document_id", "title", "document_length"]
    ].sort_values(
        "document_length",
        ascending=False
    )
)

,document_id,title,document_length
10,KK_DOC_011,Explore the post-pandemic resurgence of digita...,7868
9,KK_DOC_010,Digital marketing growth in India in 2023-Expl...,6109
0,KK_DOC_001,Kreative kudi-Transforming Ideas into Visual M...,4711
1,KK_DOC_002,Kreative kudi-Transforming Ideas into Visual M...,4711
3,KK_DOC_004,Elevate Your Brand with Graphic Design and Dig...,4239
8,KK_DOC_009,Explore our digital marketing blog,3641
6,KK_DOC_007,Empowering Your Digital Presence | Founder & CEo,3212
2,KK_DOC_003,Your Graphic Designing Institute | Hands-On Le...,2824
5,KK_DOC_006,Stay updated with our insightful digital marke...,1332
4,KK_DOC_005,Explore our company portfolio: A showcase of o...,583


In [72]:
# Remove duplicate documents based on their actual content

before = len(rag_df)

rag_df = rag_df.drop_duplicates(
    subset=["content"],
    keep="first"
).copy()

rag_df = rag_df.reset_index(drop=True)

# Recreate document IDs
rag_df["document_id"] = [
    f"KK_DOC_{i+1:03d}"
    for i in range(len(rag_df))
]

print("Documents before duplicate removal:", before)
print("Documents after duplicate removal:", len(rag_df))

display(
    rag_df[
        ["document_id", "title", "url"]
    ]
)

Documents before duplicate removal: 11
Documents after duplicate removal: 10


,document_id,title,url
0,KK_DOC_001,Kreative kudi-Transforming Ideas into Visual M...,https://www.kreativekudi.com/
1,KK_DOC_002,Your Graphic Designing Institute | Hands-On Le...,https://www.kreativekudi.com/Institute.html
2,KK_DOC_003,Elevate Your Brand with Graphic Design and Dig...,https://www.kreativekudi.com/about.html
3,KK_DOC_004,Explore our company portfolio: A showcase of o...,https://www.kreativekudi.com/portfolio.html
4,KK_DOC_005,Stay updated with our insightful digital marke...,https://www.kreativekudi.com/articles.html
5,KK_DOC_006,Empowering Your Digital Presence | Founder & CEo,https://www.kreativekudi.com/Owner.html
6,KK_DOC_007,Get in touch with us today | Contact us,https://www.kreativekudi.com/contact.html
7,KK_DOC_008,Explore our digital marketing blog,https://www.kreativekudi.com/article1.html
8,KK_DOC_009,Digital marketing growth in India in 2023-Expl...,https://www.kreativekudi.com/article2.html
9,KK_DOC_010,Explore the post-pandemic resurgence of digita...,https://www.kreativekudi.com/article3.html


In [73]:
rag_df.to_csv(
    "kreative_kudi_rag_documents.csv",
    index=False,
    encoding="utf-8"
)

print("Saved:", len(rag_df), "unique documents")

Saved: 10 unique documents


In [74]:
import pandas as pd
import re

rag_df = pd.read_csv("kreative_kudi_rag_documents.csv")

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def create_chunks(text, chunk_size=500, overlap=80, min_chunk_size=50):

    text = clean_text(text)
    words = text.split()

    chunks = []
    start = 0

    while start < len(words):

        end = min(start + chunk_size, len(words))

        chunk = " ".join(words[start:end]).strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    # Merge very small final chunk
    if len(chunks) > 1:

        last_size = len(chunks[-1].split())

        if last_size < min_chunk_size:
            chunks[-2] = chunks[-2] + " " + chunks[-1]
            chunks.pop()

    return chunks


chunks = []

for _, row in rag_df.iterrows():

    title = clean_text(row["title"])
    headings = clean_text(row["headings"])
    content = clean_text(row["content"])

    # IMPORTANT:
    # Include title + headings + content
    document_text = f"""
Title: {title}

Headings: {headings}

Content: {content}
"""

    document_chunks = create_chunks(
        document_text,
        chunk_size=500,
        overlap=80,
        min_chunk_size=50
    )

    for i, chunk in enumerate(document_chunks):

        chunks.append({
            "chunk_id": f'{row["document_id"]}_CHUNK_{i+1:03d}',
            "document_id": row["document_id"],
            "title": title,
            "source_url": row["url"],
            "chunk_number": i + 1,
            "text": chunk
        })


chunks_df = pd.DataFrame(chunks)

chunks_df["word_count"] = chunks_df["text"].apply(
    lambda x: len(str(x).split())
)

print("Total chunks:", len(chunks_df))
print(
    "Average words:",
    round(chunks_df["word_count"].mean(), 2)
)
print(
    "Minimum words:",
    chunks_df["word_count"].min()
)
print(
    "Maximum words:",
    chunks_df["word_count"].max()
)

Total chunks: 17
Average words: 342.29
Minimum words: 59
Maximum words: 500


In [75]:
chunks_df.to_csv(
    "kreative_kudi_chunks_v2.csv",
    index=False,
    encoding="utf-8"
)

print("Saved successfully.")

Saved successfully.


In [76]:
chunks = []

for _, row in rag_df.iterrows():

    document_chunks = create_chunks(
        row["content"],
        chunk_size=600,
        overlap=100
    )

    for i, chunk in enumerate(document_chunks):

        chunks.append({
            "chunk_id": f'{row["document_id"]}_CHUNK_{i+1:03d}',
            "document_id": row["document_id"],
            "title": row["title"],
            "source_url": row["url"],
            "chunk_number": i + 1,
            "text": chunk
        })


chunks_df = pd.DataFrame(chunks)

print("Total chunks:", len(chunks_df))

Total chunks: 14


In [77]:
display(
    chunks_df[
        [
            "chunk_id",
            "document_id",
            "title",
            "chunk_number",
            "text"
        ]
    ].head(10)
)

,chunk_id,document_id,title,chunk_number,text
0,KK_DOC_001_CHUNK_001,KK_DOC_001,Kreative kudi-Transforming Ideas into Visual M...,1,"In the world of Internet Customer Service, it’..."
1,KK_DOC_001_CHUNK_002,KK_DOC_001,Kreative kudi-Transforming Ideas into Visual M...,2,"provided us with a comprehensive, fast and wel..."
2,KK_DOC_002_CHUNK_001,KK_DOC_002,Your Graphic Designing Institute | Hands-On Le...,1,"Learn skills to become job ready, Best Coachin..."
3,KK_DOC_003_CHUNK_001,KK_DOC_003,Elevate Your Brand with Graphic Design and Dig...,1,"At the intersection of art, technology and bus..."
4,KK_DOC_004_CHUNK_001,KK_DOC_004,Explore our company portfolio: A showcase of o...,1,Creativity involves breaking out of expected &...
5,KK_DOC_005_CHUNK_001,KK_DOC_005,Stay updated with our insightful digital marke...,1,All the most current news and events on Digita...
6,KK_DOC_006_CHUNK_001,KK_DOC_006,Empowering Your Digital Presence | Founder & CEo,1,Years Of Experience Projects Completed In last...
7,KK_DOC_007_CHUNK_001,KK_DOC_007,Get in touch with us today | Contact us,1,Feel free to ask me any question or let's do t...
8,KK_DOC_008_CHUNK_001,KK_DOC_008,Explore our digital marketing blog,1,THE ECONOMIC TIMES A digital marketing strateg...
9,KK_DOC_009_CHUNK_001,KK_DOC_009,Digital marketing growth in India in 2023-Expl...,1,IIM SKILLS The emergence of digital marketing ...


In [78]:
chunks_df["word_count"] = chunks_df["text"].apply(
    lambda x: len(str(x).split())
)

print("Average words per chunk:",
      round(chunks_df["word_count"].mean(), 2))

print("Minimum words:",
      chunks_df["word_count"].min())

print("Maximum words:",
      chunks_df["word_count"].max())

Average words per chunk: 372.14
Minimum words: 27
Maximum words: 600


In [79]:
doc_id = "KK_DOC_001"

display(
    chunks_df[
        chunks_df["document_id"] == doc_id
    ][
        [
            "chunk_id",
            "chunk_number",
            "word_count",
            "text"
        ]
    ]
)

,chunk_id,chunk_number,word_count,text
0,KK_DOC_001_CHUNK_001,1,600,"In the world of Internet Customer Service, it’..."
1,KK_DOC_001_CHUNK_002,2,146,"provided us with a comprehensive, fast and wel..."


In [80]:
# Find very small chunks

small_chunks = chunks_df[
    chunks_df["word_count"] < 50
].copy()

print("Chunks below 50 words:", len(small_chunks))

display(
    small_chunks[
        [
            "chunk_id",
            "document_id",
            "title",
            "word_count",
            "text"
        ]
    ]
)

Chunks below 50 words: 1


,chunk_id,document_id,title,word_count,text
7,KK_DOC_007_CHUNK_001,KK_DOC_007,Get in touch with us today | Contact us,27,Feel free to ask me any question or let's do t...


In [81]:
def create_chunks(text, chunk_size=600, overlap=100, min_chunk_size=50):

    text = clean_for_chunking(text)
    words = text.split()

    chunks = []
    start = 0

    while start < len(words):

        end = min(start + chunk_size, len(words))

        chunk = " ".join(words[start:end]).strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    # Merge very small final chunks
    if len(chunks) > 1:

        last_words = len(chunks[-1].split())

        if last_words < min_chunk_size:
            chunks[-2] = chunks[-2] + " " + chunks[-1]
            chunks.pop()

    return chunks

In [86]:
def create_chunks(text, chunk_size=600, overlap=100, min_chunk_size=50):

    text = clean_text(text)
    words = text.split()

    chunks = []
    start = 0

    while start < len(words):

        end = min(start + chunk_size, len(words))

        chunk = " ".join(words[start:end]).strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    # Merge very small final chunks
    if len(chunks) > 1:

        last_words = len(chunks[-1].split())

        if last_words < min_chunk_size:
            chunks[-2] = chunks[-2] + " " + chunks[-1]
            chunks.pop()

    return chunks


chunks = []

for _, row in rag_df.iterrows():

    document_chunks = create_chunks(
        row["content"],
        chunk_size=600,
        overlap=100,
        min_chunk_size=50
    )

    for i, chunk in enumerate(document_chunks):

        chunks.append({
            "chunk_id": f'{row["document_id"]}_CHUNK_{i+1:03d}',
            "document_id": row["document_id"],
            "title": row["title"],
            "source_url": row["url"],
            "chunk_number": i + 1,
            "text": chunk
        })


chunks_df = pd.DataFrame(chunks)

chunks_df["word_count"] = chunks_df["text"].apply(
    lambda x: len(str(x).split())
)

print("Total chunks:", len(chunks_df))
print("Average words:", round(chunks_df["word_count"].mean(), 2))
print("Minimum words:", chunks_df["word_count"].min())
print("Maximum words:", chunks_df["word_count"].max())

Total chunks: 14
Average words: 372.14
Minimum words: 27
Maximum words: 600


In [87]:
chunks_df.to_csv(
    "kreative_kudi_chunks.csv",
    index=False,
    encoding="utf-8"
)

print("Chunk dataset saved successfully.")

Chunk dataset saved successfully.


In [88]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.3 MB/s eta 0:00:00


In [89]:
import pandas as pd

chunks_df = pd.read_csv("kreative_kudi_chunks.csv")

print("Total chunks:", len(chunks_df))
display(chunks_df.head())

Total chunks: 14


,chunk_id,document_id,title,source_url,chunk_number,text,word_count
0,KK_DOC_001_CHUNK_001,KK_DOC_001,Kreative kudi-Transforming Ideas into Visual M...,https://www.kreativekudi.com/,1,"In the world of Internet Customer Service, it’...",600
1,KK_DOC_001_CHUNK_002,KK_DOC_001,Kreative kudi-Transforming Ideas into Visual M...,https://www.kreativekudi.com/,2,"provided us with a comprehensive, fast and wel...",146
2,KK_DOC_002_CHUNK_001,KK_DOC_002,Your Graphic Designing Institute | Hands-On Le...,https://www.kreativekudi.com/Institute.html,1,"Learn skills to become job ready, Best Coachin...",324
3,KK_DOC_003_CHUNK_001,KK_DOC_003,Elevate Your Brand with Graphic Design and Dig...,https://www.kreativekudi.com/about.html,1,"At the intersection of art, technology and bus...",561
4,KK_DOC_004_CHUNK_001,KK_DOC_004,Explore our company portfolio: A showcase of o...,https://www.kreativekudi.com/portfolio.html,1,Creativity involves breaking out of expected &...,63


In [90]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [91]:
texts = chunks_df["text"].fillna("").tolist()

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (14, 384)


In [92]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = chunks_df["text"].fillna("").tolist()

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (14, 384)


In [93]:
embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dimension)

index.add(
    embeddings.astype("float32")
)

print("Vectors stored:", index.ntotal)

Vectors stored: 14


In [94]:
query = "What courses are available at Kreative Kudi?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

scores, indices = index.search(
    query_embedding,
    k=5
)

for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]),
    start=1
):

    row = chunks_df.iloc[idx]

    print("=" * 100)
    print(f"Rank: {rank}")
    print(f"Similarity: {score:.4f}")
    print(f"Document: {row['document_id']}")
    print(f"Title: {row['title']}")
    print(f"URL: {row['source_url']}")
    print("\nTEXT:")
    print(row["text"][:1200])

Rank: 1
Similarity: 0.4837
Document: KK_DOC_001
Title: Kreative kudi-Transforming Ideas into Visual Masterpieces
URL: https://www.kreativekudi.com/

TEXT:
provided us with a comprehensive, fast and well planned digital marketing strategy that has yielded great results in terms of content, SEO, Social Media. His team are a pleasure to work with, as well as being fast to respond and adapt to the needs of your brand. Kreative Kudi executes our entire digital marketing strategy with incredible skill and expertise. Their ability to provide consistent results and offer the latest techniques keeps our brand ahead of the competition in an ever-changing digital environment. Our area of practice is quite wide: Graphics Design, Logo Design, Branding, Digital Marketing, Lead Generation, Ui/Ux Design and many more... SATYA TWO,near Bharat Petroleum, Shastrinagar, Naranpura, Ahmedabad-380013 Gujarat, INDIA kreativekudi@gmail.com +918160581704 Digital Marketing Graphics Designing Logo Designing Video

In [95]:
query = "What courses are available at Kreative Kudi?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

scores, indices = index.search(
    query_embedding,
    k=5
)

for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]),
    start=1
):

    print("=" * 80)
    print(f"Rank: {rank}")
    print(f"Similarity: {score:.4f}")
    print(f"Title: {chunks_df.iloc[idx]['title']}")
    print("\n", chunks_df.iloc[idx]["text"][:800])

Rank: 1
Similarity: 0.4837
Title: Kreative kudi-Transforming Ideas into Visual Masterpieces

 provided us with a comprehensive, fast and well planned digital marketing strategy that has yielded great results in terms of content, SEO, Social Media. His team are a pleasure to work with, as well as being fast to respond and adapt to the needs of your brand. Kreative Kudi executes our entire digital marketing strategy with incredible skill and expertise. Their ability to provide consistent results and offer the latest techniques keeps our brand ahead of the competition in an ever-changing digital environment. Our area of practice is quite wide: Graphics Design, Logo Design, Branding, Digital Marketing, Lead Generation, Ui/Ux Design and many more... SATYA TWO,near Bharat Petroleum, Shastrinagar, Naranpura, Ahmedabad-380013 Gujarat, INDIA kreativekudi@gmail.com +918160581704 Digital Mark
Rank: 2
Similarity: 0.4315
Title: Kreative kudi-Transforming Ideas into Visual Masterpieces

 In the worl

In [97]:
row["content"]

KeyError: 'content'

In [98]:
# Create a searchable text that gives headings more importance

chunks_df["search_text"] = (
    chunks_df["title"].fillna("") + " " +
    chunks_df["title"].fillna("") + " " +
    chunks_df["text"].fillna("")
)

search_texts = chunks_df["search_text"].tolist()

search_embeddings = embedding_model.encode(
    search_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

search_embeddings = search_embeddings.astype("float32")

# New FAISS index
search_index = faiss.IndexFlatIP(
    search_embeddings.shape[1]
)

search_index.add(search_embeddings)

print("Vectors stored:", search_index.ntotal)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Vectors stored: 14


In [99]:
query = "What courses are available at Kreative Kudi?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

scores, indices = search_index.search(
    query_embedding,
    k=5
)

for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]),
    start=1
):

    row = chunks_df.iloc[idx]

    print("=" * 100)
    print(f"Rank: {rank}")
    print(f"Similarity: {score:.4f}")
    print(f"Document: {row['document_id']}")
    print(f"Title: {row['title']}")
    print(f"URL: {row['source_url']}")
    print("\nTEXT:")
    print(row["text"][:1000])

Rank: 1
Similarity: 0.5101
Document: KK_DOC_001
Title: Kreative kudi-Transforming Ideas into Visual Masterpieces
URL: https://www.kreativekudi.com/

TEXT:
provided us with a comprehensive, fast and well planned digital marketing strategy that has yielded great results in terms of content, SEO, Social Media. His team are a pleasure to work with, as well as being fast to respond and adapt to the needs of your brand. Kreative Kudi executes our entire digital marketing strategy with incredible skill and expertise. Their ability to provide consistent results and offer the latest techniques keeps our brand ahead of the competition in an ever-changing digital environment. Our area of practice is quite wide: Graphics Design, Logo Design, Branding, Digital Marketing, Lead Generation, Ui/Ux Design and many more... SATYA TWO,near Bharat Petroleum, Shastrinagar, Naranpura, Ahmedabad-380013 Gujarat, INDIA kreativekudi@gmail.com +918160581704 Digital Marketing Graphics Designing Logo Designing Video

In [100]:
def semantic_search(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = search_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        row = chunks_df.iloc[idx]

        results.append({
            "rank": rank,
            "score": float(score),
            "document_id": row["document_id"],
            "title": row["title"],
            "url": row["source_url"],
            "text": row["text"]
        })

    return results

In [101]:
test_questions = [
    "What digital marketing services does Kreative Kudi offer?",
    "What courses are available at Kreative Kudi?",
    "When was Kreative Kudi established?",
    "Who is the founder of Kreative Kudi?",
    "How can I contact Kreative Kudi?"
]

for question in test_questions:

    print("\n")
    print("#" * 100)
    print("QUESTION:", question)
    print("#" * 100)

    results = semantic_search(question, top_k=3)

    for result in results:

        print(
            f"\nRank {result['rank']} | "
            f"Score: {result['score']:.4f}"
        )

        print("Document:", result["document_id"])
        print("Title:", result["title"])
        print("URL:", result["url"])
        print("Text:", result["text"][:500])



####################################################################################################
QUESTION: What digital marketing services does Kreative Kudi offer?
####################################################################################################

Rank 1 | Score: 0.7357
Document: KK_DOC_001
Title: Kreative kudi-Transforming Ideas into Visual Masterpieces
URL: https://www.kreativekudi.com/
Text: In the world of Internet Customer Service, it’s important to remember your competitor is only one mouse click away. Logos are the graphic extension of the internal realities of a company.. The Internet makes money for you when you build something that is real and when it matters to people! Click on this tab to open WhatsApp and start a chat. Welcome to Kreative Kudi Digital Marketing Company and Institute, where we specialize in transforming your online presence and driving exceptional results. 

Rank 2 | Score: 0.6426
Document: KK_DOC_003
Title: Elevate Your Brand with 

In [102]:
# Inspect Owner and Contact documents

for doc_id in ["KK_DOC_007", "KK_DOC_008"]:

    print("\n" + "=" * 100)
    print("DOCUMENT:", doc_id)
    print("=" * 100)

    doc_chunks = chunks_df[
        chunks_df["document_id"] == doc_id
    ]

    for _, row in doc_chunks.iterrows():

        print("\nCHUNK:", row["chunk_id"])
        print(row["text"])


DOCUMENT: KK_DOC_007

CHUNK: KK_DOC_007_CHUNK_001
Feel free to ask me any question or let's do to talk about our future collaboration. SATYA TWO,near Bharat Petroleum, Shastrinagar, Naranpura, Ahmedabad-380013 Gujarat, INDIA kreativekudi@gmail.com +918160581704

DOCUMENT: KK_DOC_008

CHUNK: KK_DOC_008_CHUNK_001
THE ECONOMIC TIMES A digital marketing strategy involves using electronic channels to reach customers with products and services. This sums up the digital marketing definition. By utilising electronic devices to convey promotional messaging, marketing specialists can measure the impact of marketing on the customer journey. A digital marketing campaign refers to products and services on tablets, computer screens, phones, and other devices. Many different forms of advertising exist, including display ads, paid social ads, search engine marketing, online videos, and social media posts. Technically, SEO, or search engine optimisation, is a marketing tool rather than a marketing act

In [103]:
# Show all documents and their URLs

display(
    rag_df[
        ["document_id", "title", "url"]
    ].sort_values("document_id")
)

,document_id,title,url
0,KK_DOC_001,Kreative kudi-Transforming Ideas into Visual M...,https://www.kreativekudi.com/
1,KK_DOC_002,Your Graphic Designing Institute | Hands-On Le...,https://www.kreativekudi.com/Institute.html
2,KK_DOC_003,Elevate Your Brand with Graphic Design and Dig...,https://www.kreativekudi.com/about.html
3,KK_DOC_004,Explore our company portfolio: A showcase of o...,https://www.kreativekudi.com/portfolio.html
4,KK_DOC_005,Stay updated with our insightful digital marke...,https://www.kreativekudi.com/articles.html
5,KK_DOC_006,Empowering Your Digital Presence | Founder & CEo,https://www.kreativekudi.com/Owner.html
6,KK_DOC_007,Get in touch with us today | Contact us,https://www.kreativekudi.com/contact.html
7,KK_DOC_008,Explore our digital marketing blog,https://www.kreativekudi.com/article1.html
8,KK_DOC_009,Digital marketing growth in India in 2023-Expl...,https://www.kreativekudi.com/article2.html
9,KK_DOC_010,Explore the post-pandemic resurgence of digita...,https://www.kreativekudi.com/article3.html


In [104]:
import pandas as pd

rag_df = pd.read_csv("kreative_kudi_rag_documents.csv")

def get_page_type(url):

    url = str(url).lower()

    if "institute.html" in url:
        return "training"

    elif "about.html" in url:
        return "company_services"

    elif "portfolio.html" in url:
        return "portfolio"

    elif "articles.html" in url:
        return "articles"

    elif "owner.html" in url:
        return "founder"

    elif "contact.html" in url:
        return "contact"

    elif "article1.html" in url:
        return "article"

    elif "article2.html" in url:
        return "article"

    elif "article3.html" in url:
        return "article"

    elif url.rstrip("/") == "https://www.kreativekudi.com":
        return "home"

    else:
        return "other"


rag_df["page_type"] = rag_df["url"].apply(get_page_type)

display(
    rag_df[
        ["document_id", "title", "page_type", "url"]
    ]
)

,document_id,title,page_type,url
0,KK_DOC_001,Kreative kudi-Transforming Ideas into Visual M...,home,https://www.kreativekudi.com/
1,KK_DOC_002,Your Graphic Designing Institute | Hands-On Le...,training,https://www.kreativekudi.com/Institute.html
2,KK_DOC_003,Elevate Your Brand with Graphic Design and Dig...,company_services,https://www.kreativekudi.com/about.html
3,KK_DOC_004,Explore our company portfolio: A showcase of o...,portfolio,https://www.kreativekudi.com/portfolio.html
4,KK_DOC_005,Stay updated with our insightful digital marke...,articles,https://www.kreativekudi.com/articles.html
5,KK_DOC_006,Empowering Your Digital Presence | Founder & CEo,founder,https://www.kreativekudi.com/Owner.html
6,KK_DOC_007,Get in touch with us today | Contact us,contact,https://www.kreativekudi.com/contact.html
7,KK_DOC_008,Explore our digital marketing blog,article,https://www.kreativekudi.com/article1.html
8,KK_DOC_009,Digital marketing growth in India in 2023-Expl...,article,https://www.kreativekudi.com/article2.html
9,KK_DOC_010,Explore the post-pandemic resurgence of digita...,article,https://www.kreativekudi.com/article3.html


In [105]:
# Create mapping from document_id → page_type

page_type_map = dict(
    zip(
        rag_df["document_id"],
        rag_df["page_type"]
    )
)

chunks_df["page_type"] = chunks_df["document_id"].map(
    page_type_map
)

display(
    chunks_df[
        [
            "chunk_id",
            "document_id",
            "page_type",
            "title"
        ]
    ].head(20)
)

,chunk_id,document_id,page_type,title
0,KK_DOC_001_CHUNK_001,KK_DOC_001,home,Kreative kudi-Transforming Ideas into Visual M...
1,KK_DOC_001_CHUNK_002,KK_DOC_001,home,Kreative kudi-Transforming Ideas into Visual M...
2,KK_DOC_002_CHUNK_001,KK_DOC_002,training,Your Graphic Designing Institute | Hands-On Le...
3,KK_DOC_003_CHUNK_001,KK_DOC_003,company_services,Elevate Your Brand with Graphic Design and Dig...
4,KK_DOC_004_CHUNK_001,KK_DOC_004,portfolio,Explore our company portfolio: A showcase of o...
5,KK_DOC_005_CHUNK_001,KK_DOC_005,articles,Stay updated with our insightful digital marke...
6,KK_DOC_006_CHUNK_001,KK_DOC_006,founder,Empowering Your Digital Presence | Founder & CEo
7,KK_DOC_007_CHUNK_001,KK_DOC_007,contact,Get in touch with us today | Contact us
8,KK_DOC_008_CHUNK_001,KK_DOC_008,article,Explore our digital marketing blog
9,KK_DOC_009_CHUNK_001,KK_DOC_009,article,Digital marketing growth in India in 2023-Expl...


In [106]:
chunks_df.to_csv(
    "kreative_kudi_chunks_v3.csv",
    index=False,
    encoding="utf-8"
)

print("Saved successfully.")
print("Total chunks:", len(chunks_df))

Saved successfully.
Total chunks: 14


In [120]:
import re
import numpy as np
from collections import Counter

# ---------------------------------------------------------
# 1. Prepare searchable text
# ---------------------------------------------------------

chunks_df["search_text"] = (
    chunks_df["title"].fillna("") + " " +
    chunks_df["page_type"].fillna("") + " " +
    chunks_df["text"].fillna("")
).str.lower()


# ---------------------------------------------------------
# 2. Simple keyword scoring
# ---------------------------------------------------------

import re
from collections import Counter

QUERY_EXPANSIONS = {

    "course": [
        "course",
        "courses",
        "training",
        "coaching",
        "class",
        "classes",
        "program",
        "programs",
        "learn",
        "learning",
        "education",
        "institute"
    ],

    "founder": [
        "founder",
        "owner",
        "ceo",
        "chief executive",
        "started",
        "established"
    ],

    "contact": [
        "contact",
        "email",
        "phone",
        "mobile",
        "telephone",
        "address",
        "location",
        "reach",
        "talk",
        "get in touch"
    ],

    "portfolio": [
        "portfolio",
        "project",
        "projects",
        "work",
        "clients",
        "case study"
    ],

    "services": [
        "service",
        "services",
        "seo",
        "marketing",
        "branding",
        "design",
        "graphic",
        "logo",
        "ui",
        "ux",
        "social media",
        "lead generation"
    ]
}


def expand_query(query):

    query = query.lower()

    expanded_terms = set(
        re.findall(r"\b[a-zA-Z]{2,}\b", query)
    )

    for key, synonyms in QUERY_EXPANSIONS.items():

        if key in query:

            expanded_terms.update(synonyms)

    return list(expanded_terms)


def keyword_score(query, text):

    query_terms = expand_query(query)

    text_words = re.findall(
        r"\b[a-zA-Z]{2,}\b",
        text.lower()
    )

    text_counter = Counter(text_words)

    if not query_terms:
        return 0.0

    matched = sum(
        1
        for term in query_terms
        if term in text_counter
    )

    return matched / len(query_terms)


# ---------------------------------------------------------
# 3. Detect query intent
# ---------------------------------------------------------

def detect_page_type(query):

    query = query.lower().strip()

    # Training / Courses
    training_terms = [
        "course", "courses", "training", "institute",
        "learn", "learning", "class", "classes",
        "education", "study", "curriculum",
        "certification", "certificate"
    ]

    # Founder / Owner
    founder_terms = [
        "founder", "owner", "ceo",
        "chief executive", "started by"
    ]

    # Company history / establishment
    company_history_terms = [
        "established",
        "establishment",
        "founded",
        "founded in",
        "started",
        "started in",
        "since",
        "how long",
        "history",
        "when did",
        "when was",
        "year company started",
        "company age"
    ]

    # Contact
    contact_terms = [
        "contact", "email", "e-mail", "phone",
        "telephone", "mobile", "number",
        "address", "location", "reach",
        "get in touch", "talk to"
    ]

    # Portfolio
    portfolio_terms = [
        "portfolio", "projects", "project",
        "previous work", "our work",
        "case study", "case studies", "clients"
    ]

    # Services
    service_terms = [
        "service", "services", "seo",
        "digital marketing", "graphic design",
        "ui", "ux", "branding", "logo",
        "lead generation", "social media"
    ]

    if any(term in query for term in training_terms):
        return "training"

    if any(term in query for term in founder_terms):
        return "founder"

    # IMPORTANT:
    # Company establishment/history information
    # is currently present on the About page.
    if any(term in query for term in company_history_terms):
        return "company_services"

    if any(term in query for term in contact_terms):
        return "contact"

    if any(term in query for term in portfolio_terms):
        return "portfolio"

    if any(term in query for term in service_terms):
        return "company_services"

    return None

# ---------------------------------------------------------
# 4. Hybrid retrieval
# ---------------------------------------------------------

def hybrid_search(query, top_k=5):

    # -----------------------------------------
    # Semantic search
    # -----------------------------------------

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = search_index.search(
        query_embedding,
        min(10, len(chunks_df))
    )

    expected_page_type = detect_page_type(query)

    results = []

    for semantic_score, idx in zip(scores[0], indices[0]):

        row = chunks_df.iloc[idx]

        # -----------------------------------------
        # Keyword score
        # -----------------------------------------

        kw_score = keyword_score(
            query,
            row["search_text"]
        )

        # -----------------------------------------
        # Page-type score
        # -----------------------------------------

        page_boost = 0.0

        if (
            expected_page_type is not None
            and row["page_type"] == expected_page_type
        ):
            page_boost = 0.25

        # -----------------------------------------
        # Final score
        # -----------------------------------------

        final_score = (
            0.60 * float(semantic_score)
            + 0.15 * kw_score
            + page_boost
        )

        results.append({
            "semantic_score": float(semantic_score),
            "keyword_score": kw_score,
            "page_boost": page_boost,
            "final_score": final_score,
            "document_id": row["document_id"],
            "page_type": row["page_type"],
            "title": row["title"],
            "url": row["source_url"],
            "text": row["text"]
        })

    results = sorted(
        results,
        key=lambda x: x["final_score"],
        reverse=True
    )

    return results[:top_k]

In [118]:
query = "What courses are available at Kreative Kudi?"

print("Expanded query:")
print(expand_query(query))

print("\nInstitute keyword score:")

institute_row = chunks_df[
    chunks_df["document_id"] == "KK_DOC_002"
].iloc[0]

print(
    keyword_score(
        query,
        institute_row["search_text"]
    )
)

Expanded query:
['kudi', 'at', 'courses', 'course', 'kreative', 'what', 'are', 'coaching', 'education', 'class', 'programs', 'program', 'classes', 'training', 'institute', 'learning', 'learn', 'available']

Institute keyword score:
0.2777777777777778


In [119]:
test_questions = [
    "What digital marketing services does Kreative Kudi offer?",
    "What courses are available at Kreative Kudi?",
    "When was Kreative Kudi established?",
    "Who is the founder of Kreative Kudi?",
    "How can I contact Kreative Kudi?"
]

for question in test_questions:

    print("\n" + "#" * 100)
    print("QUESTION:", question)
    print("#" * 100)

    results = hybrid_search(
        question,
        top_k=3
    )

    for rank, result in enumerate(results, 1):

        print("\n" + "-" * 80)
        print(
            f"Rank {rank} | "
            f"Final: {result['final_score']:.4f} | "
            f"Semantic: {result['semantic_score']:.4f} | "
            f"Keyword: {result['keyword_score']:.4f} | "
            f"Boost: {result['page_boost']:.2f}"
        )

        print("Document:", result["document_id"])
        print("Page type:", result["page_type"])
        print("Title:", result["title"])
        print("URL:", result["url"])
        print("\n", result["text"][:700])


####################################################################################################
QUESTION: What digital marketing services does Kreative Kudi offer?
####################################################################################################

--------------------------------------------------------------------------------
Rank 1 | Final: 0.7356 | Semantic: 0.6426 | Keyword: 0.6667 | Boost: 0.25
Document: KK_DOC_003
Page type: company_services
Title: Elevate Your Brand with Graphic Design and Digital Marketing
URL: https://www.kreativekudi.com/about.html

 At the intersection of art, technology and business, we create design solutions geared toward business success. Since 2020, we have been leaders in design technology. Our services range from experience designing, identity development, product strategy, branding, user experience design, Lead generation, user interface design, UI/UX Design, Graphic Design , Logo Design,Digitak Marketing and SEO. Our work has

In [116]:
test_intents = [
    "What courses are available at Kreative Kudi?",
    "What training does Kreative Kudi provide?",
    "Who is the founder of Kreative Kudi?",
    "How can I contact Kreative Kudi?",
    "What is the email address?",
    "Show me the portfolio",
    "What digital marketing services do you offer?",
    "What is Kreative Kudi?"
]

for q in test_intents:

    print(
        f"{q:<60} → {detect_page_type(q)}"
    )

What courses are available at Kreative Kudi?                 → training
What training does Kreative Kudi provide?                    → training
Who is the founder of Kreative Kudi?                         → founder
How can I contact Kreative Kudi?                             → contact
What is the email address?                                   → contact
Show me the portfolio                                        → portfolio
What digital marketing services do you offer?                → company_services
What is Kreative Kudi?                                       → None


In [121]:
queries = [
    "When was Kreative Kudi established?",
    "When was Kreative Kudi founded?",
    "How long has Kreative Kudi been operating?",
    "When did the company start?",
    "What year was Kreative Kudi started?"
]

for q in queries:
    print(q, "→", detect_page_type(q))

When was Kreative Kudi established? → company_services
When was Kreative Kudi founded? → company_services
How long has Kreative Kudi been operating? → company_services
When did the company start? → company_services
What year was Kreative Kudi started? → company_services


In [122]:
results = hybrid_search(
    "When was Kreative Kudi established?",
    top_k=3
)

for rank, result in enumerate(results, 1):
    print(
        rank,
        result["document_id"],
        result["page_type"],
        result["final_score"]
    )

1 KK_DOC_003 company_services 0.5641782426834107
2 KK_DOC_001 home 0.38697411060333253
3 KK_DOC_001 home 0.3543020224571228
